# OCR Gate 2 — recover partial Vintern checkpoint and build manual review

This recovery notebook performs **no OCR inference and no Gemini calls**. It validates the saved checkpoint (`4,164` EasyOCR frames, `34,335` Vintern candidates, `27,927` completed Vintern results), excludes unfinished Vintern candidates, regenerates only the selected review crops from the attached keyframe Dataset, and exports a deterministic 250-row manual-review package (50 rows per video).

Kaggle setup: CPU is sufficient; GPU and Internet may remain off. Attach `thvu165/aic-2026-keyframes`. Run the setup cell, select the local `ocr_gate2_resume_checkpoint.zip` when the upload widget appears, then run the remaining cells in order. This notebook never marks Gate 2 PASS automatically.


In [ ]:
from __future__ import annotations

import csv
import hashlib
import json
import math
import shutil
import zipfile
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path

import ipywidgets as widgets
from IPython.display import FileLink, display
from PIL import Image, ImageDraw, ImageFont, ImageOps

INPUT_ROOT = Path('/kaggle/input')
OUTPUT_ROOT = Path('/kaggle/working/ocr-gate2-partial-review')
RECOVERED_ROOT = OUTPUT_ROOT / 'checkpoint'
SHEET_ROOT = OUTPUT_ROOT / 'review-sheets'
CHECKPOINT_NAME = 'ocr_gate2_resume_checkpoint.zip'
REVIEW_CSV = Path('/kaggle/working/ocr_gate2_partial_manual_review.csv')
REVIEW_SHEETS_ZIP = Path('/kaggle/working/ocr_gate2_partial_manual_review_sheets.zip')
SUMMARY_JSON = Path('/kaggle/working/ocr_gate2_partial_summary.json')
ARTIFACT_ZIP = Path('/kaggle/working/ocr_gate2_partial_review_artifacts.zip')
MANIFEST_JSON = OUTPUT_ROOT / 'checkpoint-manifest.json'

DEV_VIDEO_IDS = ['L21_V001', 'L21_V002', 'L21_V003', 'L21_V005', 'L21_V006']
EXPECTED = {
    'easyocr-frames.jsonl': {
        'records': 4164,
        'sha256': 'e13a4fff27401f896af9f37ab180895ad5f35fb7dbaeb656e4c3882453842a46',
        'id_field': 'keyframe_uid',
    },
    'vintern-candidates.jsonl': {
        'records': 34335,
        'sha256': 'c54cb410f3fd1787d832a9713fbf069b098948a2f545f2c19720935e2f844ce9',
        'id_field': 'candidate_id',
    },
    'vintern-results.jsonl': {
        'records': 27927,
        'sha256': '5dd3a6eb772f2c3ca5b46125e0249403c0f9a360f167f33986bd5793b28f6c99',
        'id_field': 'candidate_id',
    },
}

for directory in (OUTPUT_ROOT, RECOVERED_ROOT, SHEET_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

def load_jsonl(path, id_field):
    rows = []
    seen = set()
    with Path(path).open('r', encoding='utf-8') as handle:
        for line_number, line in enumerate(handle, 1):
            if not line.strip():
                continue
            row = json.loads(line)
            identity = row[id_field]
            if identity in seen:
                raise ValueError(f'duplicate {id_field} at line {line_number}: {identity}')
            seen.add(identity)
            rows.append(row)
    return rows

def atomic_json(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')
    temporary.replace(path)

def stable_order(rows, salt):
    return sorted(
        rows,
        key=lambda row: hashlib.sha256(f"{salt}|{row['review_id']}".encode()).hexdigest(),
    )

print('READY', OUTPUT_ROOT)


In [ ]:
# Reuse a ZIP already attached as Kaggle Input/Working; otherwise display a local upload widget.
search_patterns = [
    CHECKPOINT_NAME,
    f'*/{CHECKPOINT_NAME}',
    f'*/*/{CHECKPOINT_NAME}',
    f'*/*/*/{CHECKPOINT_NAME}',
]
zip_candidates = []
for root in (Path('/kaggle/working'), INPUT_ROOT):
    for pattern in search_patterns:
        zip_candidates.extend(path for path in root.glob(pattern) if path.is_file())
zip_candidates = sorted(set(zip_candidates), key=lambda path: str(path))

checkpoint_uploader = None
if zip_candidates:
    print('FOUND_CHECKPOINT', zip_candidates[0])
else:
    print('Select the 8.6 MiB ocr_gate2_resume_checkpoint.zip from your computer:')
    checkpoint_uploader = widgets.FileUpload(accept='.zip', multiple=False)
    display(checkpoint_uploader)


In [ ]:
# Resolve the selected ZIP, extract only the three allowlisted JSONLs, and validate exact bytes.
if zip_candidates:
    source_zip = zip_candidates[0]
else:
    value = checkpoint_uploader.value if checkpoint_uploader is not None else None
    if not value:
        raise RuntimeError('No checkpoint selected. Choose the ZIP in the upload widget, then rerun this cell.')
    if isinstance(value, dict):
        uploaded_name, metadata = next(iter(value.items()))
    else:
        metadata = value[0]
        uploaded_name = metadata.get('name', CHECKPOINT_NAME)
    if not str(uploaded_name).lower().endswith('.zip'):
        raise ValueError(f'Expected a ZIP, received: {uploaded_name}')
    source_zip = OUTPUT_ROOT / CHECKPOINT_NAME
    source_zip.write_bytes(bytes(metadata['content']))

with zipfile.ZipFile(source_zip) as archive:
    members_by_basename = {Path(member).name: member for member in archive.namelist() if not member.endswith('/')}
    missing = sorted(set(EXPECTED) - set(members_by_basename))
    if missing:
        raise RuntimeError(f'Checkpoint ZIP missing required files: {missing}')
    for filename in EXPECTED:
        target = RECOVERED_ROOT / filename
        target.write_bytes(archive.read(members_by_basename[filename]))

loaded = {}
manifest_files = {}
for filename, expected in EXPECTED.items():
    path = RECOVERED_ROOT / filename
    actual_hash = sha256_file(path)
    if actual_hash != expected['sha256']:
        raise RuntimeError(f'{filename} SHA-256 mismatch: {actual_hash}')
    rows = load_jsonl(path, expected['id_field'])
    if len(rows) != expected['records']:
        raise RuntimeError(f"{filename} record mismatch: {len(rows)} != {expected['records']}")
    loaded[filename] = rows
    manifest_files[filename] = {
        'records': len(rows),
        'bytes': path.stat().st_size,
        'sha256': actual_hash,
    }

easy_rows = loaded['easyocr-frames.jsonl']
candidate_rows = loaded['vintern-candidates.jsonl']
vintern_rows = loaded['vintern-results.jsonl']
easy_uids = {row['keyframe_uid'] for row in easy_rows}
candidate_ids = {row['candidate_id'] for row in candidate_rows}
result_ids = {row['candidate_id'] for row in vintern_rows}
if not result_ids <= candidate_ids:
    raise RuntimeError(f'Foreign Vintern result IDs: {len(result_ids - candidate_ids)}')
if any(row['keyframe_uid'] not in easy_uids for row in candidate_rows):
    raise RuntimeError('A Vintern candidate references an unknown keyframe_uid')
if Counter(row['video_id'] for row in easy_rows) != Counter({
    'L21_V001': 1008, 'L21_V002': 843, 'L21_V003': 765, 'L21_V005': 744, 'L21_V006': 804
}):
    raise RuntimeError('EasyOCR dev-subset-5 video counts do not match the audited catalog')

candidate_by_video = Counter(row['video_id'] for row in candidate_rows)
result_by_video = Counter(row['video_id'] for row in vintern_rows)
coverage = {
    video_id: {
        'completed': result_by_video[video_id],
        'candidates': candidate_by_video[video_id],
        'fraction': result_by_video[video_id] / max(1, candidate_by_video[video_id]),
    }
    for video_id in DEV_VIDEO_IDS
}
if any(item['completed'] < 50 for item in coverage.values()):
    raise RuntimeError(f'Insufficient completed Vintern rows for balanced review: {coverage}')

checkpoint_manifest = {
    'schema_version': 1,
    'created_utc': utc_now(),
    'source_zip': str(source_zip),
    'files': manifest_files,
    'vintern_coverage_by_video': coverage,
    'validation': {
        'json_valid': True,
        'duplicate_ids': 0,
        'foreign_result_ids': 0,
        'candidate_uids_exist_in_easyocr': True,
    },
}
atomic_json(MANIFEST_JSON, checkpoint_manifest)
print(json.dumps(checkpoint_manifest, ensure_ascii=False, indent=2))


In [ ]:
# Build a deterministic balanced review pool. Unfinished Vintern candidates are excluded, never relabeled as EasyOCR pass.
vintern_by_id = {row['candidate_id']: row for row in vintern_rows}
frame_by_uid = {row['keyframe_uid']: row for row in easy_rows}
review_pool = []
unfinished_regions = 0

for frame in easy_rows:
    if frame['status'] == 'no_text':
        review_pool.append({
            'review_id': hashlib.sha256(f"review-no-text|{frame['keyframe_uid']}".encode()).hexdigest()[:24],
            'video_id': frame['video_id'],
            'keyframe_uid': frame['keyframe_uid'],
            'region_id': '',
            'route': 'craft_no_text_control',
            'source_image': frame['source_image'],
            'source_path': frame['source_path'],
            'bbox_px': '',
            'escalation_reasons': '',
            'easyocr_text': '',
            'easyocr_confidence': '',
            'vintern_text': '',
            'vintern_status': '',
        })

    for region in frame.get('regions', []):
        needs_vintern = bool(region.get('escalation_reasons'))
        vintern = vintern_by_id.get(region['region_id'])
        if needs_vintern and vintern is None:
            unfinished_regions += 1
            continue
        if not needs_vintern:
            route = 'easyocr_pass'
        elif vintern.get('gemini_residual_reasons'):
            route = 'gemini_residual'
        else:
            route = 'vintern_pass'
        review_pool.append({
            'review_id': hashlib.sha256(f"review|{region['region_id']}".encode()).hexdigest()[:24],
            'video_id': frame['video_id'],
            'keyframe_uid': frame['keyframe_uid'],
            'region_id': region['region_id'],
            'route': route,
            'source_image': frame['source_image'],
            'source_path': frame['source_path'],
            'bbox_px': json.dumps(region['bbox_px'], separators=(',', ':')),
            'escalation_reasons': '|'.join(region.get('escalation_reasons', [])),
            'easyocr_text': region.get('easyocr_text', ''),
            'easyocr_confidence': region.get('easyocr_confidence', ''),
            'vintern_text': vintern.get('vintern_text', '') if vintern else '',
            'vintern_status': vintern.get('status', '') if vintern else '',
        })

route_quotas = [
    ('craft_no_text_control', 10),
    ('gemini_residual', 10),
    ('vintern_pass', 15),
    ('easyocr_pass', 15),
]
review = []
for video_id in DEV_VIDEO_IDS:
    video_pool = [row for row in review_pool if row['video_id'] == video_id]
    chosen = []
    chosen_ids = set()
    for route, quota in route_quotas:
        route_pool = stable_order([row for row in video_pool if row['route'] == route], f'{video_id}|{route}')
        for row in route_pool[:quota]:
            if row['review_id'] not in chosen_ids:
                chosen.append(row)
                chosen_ids.add(row['review_id'])
    fill_pool = stable_order([row for row in video_pool if row['review_id'] not in chosen_ids], f'{video_id}|fill')
    chosen.extend(fill_pool[:max(0, 50 - len(chosen))])
    if len(chosen) != 50:
        raise RuntimeError(f'{video_id} has only {len(chosen)} eligible review rows')
    review.extend(chosen)

if len(review) != 250 or len({row['review_id'] for row in review}) != 250:
    raise RuntimeError('Review sample must contain exactly 250 unique rows')
print('REVIEW_SAMPLE', len(review), 'UNFINISHED_EXCLUDED', unfinished_regions)
print(json.dumps({video_id: Counter(row['route'] for row in review if row['video_id'] == video_id) for video_id in DEV_VIDEO_IDS}, indent=2))


In [ ]:
# Resolve source JPEGs, regenerate the 250 selected crops, and package review artifacts.
batch_roots = set()
for pattern in ('keyframes-batch-*', '*/keyframes-batch-*', '*/*/keyframes-batch-*', '*/*/*/keyframes-batch-*'):
    batch_roots.update(path for path in INPUT_ROOT.glob(pattern) if path.is_dir())
batch_roots = sorted(batch_roots, key=lambda path: str(path))

def resolve_source_image(frame):
    direct_candidates = [Path(frame['source_path']), INPUT_ROOT / frame['source_image']]
    for candidate in direct_candidates:
        if candidate.is_file():
            return candidate
    source_parts = Path(frame['source_image']).parts
    batch_name = next((part for part in source_parts if part.startswith('keyframes-batch-')), None)
    filename = Path(frame['source_image']).name
    if batch_name:
        for batch_root in batch_roots:
            if batch_root.name != batch_name:
                continue
            candidate = batch_root / frame['video_id'] / filename
            if candidate.is_file():
                return candidate
    raise FileNotFoundError(f"Cannot resolve source image for {frame['video_id']} / {filename}. Attach thvu165/aic-2026-keyframes.")

def review_crop(row):
    frame = frame_by_uid[row['keyframe_uid']]
    source_path = resolve_source_image(frame)
    with Image.open(source_path) as opened:
        image = ImageOps.exif_transpose(opened).convert('RGB')
    if not row['region_id']:
        return image
    bbox = json.loads(row['bbox_px'])
    xs = bbox[0::2]
    ys = bbox[1::2]
    x1 = max(0, int(math.floor(min(xs))) - 4)
    y1 = max(0, int(math.floor(min(ys))) - 4)
    x2 = min(image.width, int(math.ceil(max(xs))) + 4)
    y2 = min(image.height, int(math.ceil(max(ys))) + 4)
    if x2 <= x1 or y2 <= y1:
        raise RuntimeError(f"Invalid bbox for {row['review_id']}: {bbox}")
    return image.crop((x1, y1, x2, y2))

if SHEET_ROOT.exists():
    shutil.rmtree(SHEET_ROOT)
SHEET_ROOT.mkdir(parents=True)
font_path = Path('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf')
font = ImageFont.truetype(str(font_path), 14) if font_path.exists() else ImageFont.load_default()

for sheet_start in range(0, len(review), 16):
    subset = review[sheet_start:sheet_start + 16]
    sheet_name = f'review-{sheet_start // 16:02d}.jpg'
    canvas = Image.new('RGB', (1600, 1200), 'white')
    draw = ImageDraw.Draw(canvas)
    for offset, row in enumerate(subset):
        column = offset % 4
        line = offset // 4
        x = column * 400
        y = line * 300
        crop = review_crop(row)
        crop.thumbnail((380, 175))
        canvas.paste(crop, (x + 10, y + 5))
        confidence = f"{row['easyocr_confidence']:.2f}" if isinstance(row['easyocr_confidence'], (int, float)) else '-'
        easy_text = str(row['easyocr_text']).replace('\n', ' ')[:38]
        vintern_text = str(row['vintern_text']).replace('\n', ' ')[:38]
        label = f"#{sheet_start + offset + 1} {row['video_id']} {row['route']}\nE: {easy_text} ({confidence})\nV: {vintern_text}"
        draw.multiline_text((x + 10, y + 190), label, fill='black', font=font, spacing=2)
        row['sheet_file'] = sheet_name
        row['sheet_slot'] = offset + 1
    canvas.save(SHEET_ROOT / sheet_name, quality=92)

for row in review:
    row['human_text'] = ''
    row['craft_correct'] = ''
    row['easyocr_correct'] = ''
    row['vintern_correct'] = ''
    row['preferred_engine'] = ''
    row['notes'] = ''

fieldnames = [
    'review_id', 'video_id', 'keyframe_uid', 'region_id', 'route', 'sheet_file', 'sheet_slot',
    'source_image', 'bbox_px', 'escalation_reasons', 'easyocr_text', 'easyocr_confidence',
    'vintern_text', 'vintern_status', 'human_text', 'craft_correct', 'easyocr_correct',
    'vintern_correct', 'preferred_engine', 'notes',
]
with REVIEW_CSV.open('w', encoding='utf-8-sig', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(review)

with zipfile.ZipFile(REVIEW_SHEETS_ZIP, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(SHEET_ROOT.glob('*.jpg')):
        archive.write(path, arcname=path.name)

review_counts = {
    video_id: dict(Counter(row['route'] for row in review if row['video_id'] == video_id))
    for video_id in DEV_VIDEO_IDS
}
summary = {
    'schema_version': 1,
    'created_utc': utc_now(),
    'decision': 'PARTIAL_VINTERN_CANARY_PENDING_MANUAL_ACCURACY',
    'scope': 'dev_subset_5_no_new_inference_no_gemini',
    'checkpoint': checkpoint_manifest,
    'sampling': {
        'rows': len(review),
        'rows_per_video': 50,
        'by_video_and_route': review_counts,
        'unfinished_vintern_regions_excluded': unfinished_regions,
        'deterministic': True,
    },
    'limitations': [
        'Vintern inference is partial: four videos are complete and L21_V006 is partially complete.',
        'Manual labels are required; EasyOCR/Vintern agreement is not ground-truth accuracy.',
        'This dev-only report does not change the production OCR contract or mark Gate 2 PASS.',
    ],
    'artifacts': {
        'review_csv': str(REVIEW_CSV),
        'review_sheets_zip': str(REVIEW_SHEETS_ZIP),
    },
}
atomic_json(SUMMARY_JSON, summary)
summary['artifacts']['review_csv_sha256'] = sha256_file(REVIEW_CSV)
summary['artifacts']['review_sheets_zip_sha256'] = sha256_file(REVIEW_SHEETS_ZIP)
atomic_json(SUMMARY_JSON, summary)

with zipfile.ZipFile(ARTIFACT_ZIP, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in (SUMMARY_JSON, REVIEW_CSV, REVIEW_SHEETS_ZIP, MANIFEST_JSON):
        archive.write(path, arcname=path.name)

print('PARTIAL_REVIEW_READY')
print(json.dumps(summary, ensure_ascii=False, indent=2))
display(FileLink(str(ARTIFACT_ZIP)))


## Manual labeling and accuracy

Download `ocr_gate2_partial_review_artifacts.zip`. Open the review sheets and fill the CSV columns `craft_correct`, `easyocr_correct`, and `vintern_correct` with `yes` or `no`; fill `human_text`, `preferred_engine`, and `notes` when useful. Save the edited file as `ocr_gate2_partial_manual_review_filled.csv`, then use the two cells below to calculate label coverage and accuracy. The calculator reports evidence only; it does not declare Gate 2 PASS.


In [ ]:
filled_candidates = []
for root in (Path('/kaggle/working'), INPUT_ROOT):
    for pattern in ('ocr_gate2_partial_manual_review_filled.csv', '*/ocr_gate2_partial_manual_review_filled.csv', '*/*/ocr_gate2_partial_manual_review_filled.csv'):
        filled_candidates.extend(path for path in root.glob(pattern) if path.is_file())
filled_candidates = sorted(set(filled_candidates), key=lambda path: str(path))
filled_uploader = None
if filled_candidates:
    print('FOUND_FILLED_CSV', filled_candidates[0])
else:
    print('Optional: select ocr_gate2_partial_manual_review_filled.csv after manual labeling.')
    filled_uploader = widgets.FileUpload(accept='.csv', multiple=False)
    display(filled_uploader)


In [ ]:
# Run this cell only after supplying the manually filled CSV above.
filled_path = filled_candidates[0] if filled_candidates else None
if filled_path is None and filled_uploader is not None and filled_uploader.value:
    value = filled_uploader.value
    if isinstance(value, dict):
        uploaded_name, metadata = next(iter(value.items()))
    else:
        metadata = value[0]
        uploaded_name = metadata.get('name', 'ocr_gate2_partial_manual_review_filled.csv')
    filled_path = Path('/kaggle/working/ocr_gate2_partial_manual_review_filled.csv')
    filled_path.write_bytes(bytes(metadata['content']))

if filled_path is None:
    print('PENDING_MANUAL_LABELS: download the review package, label the CSV, then rerun the two accuracy cells.')
else:
    with filled_path.open('r', encoding='utf-8-sig', newline='') as handle:
        labeled_rows = list(csv.DictReader(handle))
    if len(labeled_rows) != 250 or len({row['review_id'] for row in labeled_rows}) != 250:
        raise RuntimeError('Filled CSV must contain the same 250 unique review_id rows')
    expected_review_ids = {row['review_id'] for row in review}
    if {row['review_id'] for row in labeled_rows} != expected_review_ids:
        raise RuntimeError('Filled CSV review_id set differs from the generated sample')

    true_values = {'1', 'true', 'yes', 'y', 'correct', 'đúng', 'dung'}
    false_values = {'0', 'false', 'no', 'n', 'incorrect', 'sai'}
    def parse_label(value):
        normalized = str(value or '').strip().casefold()
        if normalized in true_values:
            return True
        if normalized in false_values:
            return False
        return None

    def metric(column, eligible):
        values = [parse_label(row[column]) for row in labeled_rows if eligible(row)]
        labeled = [value for value in values if value is not None]
        return {
            'eligible': len(values),
            'labeled': len(labeled),
            'correct': sum(value is True for value in labeled),
            'accuracy': (sum(value is True for value in labeled) / len(labeled)) if labeled else None,
        }

    metrics = {
        'craft_detection': metric('craft_correct', lambda row: True),
        'easyocr_recognition': metric('easyocr_correct', lambda row: bool(row['region_id'])),
        'vintern_recognition': metric('vintern_correct', lambda row: row['route'] in {'vintern_pass', 'gemini_residual'}),
    }
    complete = all(item['eligible'] == item['labeled'] for item in metrics.values())
    accuracy_report = {
        'schema_version': 1,
        'created_utc': utc_now(),
        'status': 'MANUAL_LABELS_COMPLETE_PENDING_USER_GATE_DECISION' if complete else 'PENDING_MANUAL_LABELS',
        'source_csv': str(filled_path),
        'source_csv_sha256': sha256_file(filled_path),
        'metrics': metrics,
        'by_video': {
            video_id: {
                'review_rows': sum(row['video_id'] == video_id for row in labeled_rows),
                'craft': metric('craft_correct', lambda row, video_id=video_id: row['video_id'] == video_id),
                'easyocr': metric('easyocr_correct', lambda row, video_id=video_id: row['video_id'] == video_id and bool(row['region_id'])),
                'vintern': metric('vintern_correct', lambda row, video_id=video_id: row['video_id'] == video_id and row['route'] in {'vintern_pass', 'gemini_residual'}),
            }
            for video_id in DEV_VIDEO_IDS
        },
        'note': 'Evidence report only; no automatic Gate 2 PASS decision.',
    }
    accuracy_path = Path('/kaggle/working/ocr_gate2_partial_accuracy.json')
    atomic_json(accuracy_path, accuracy_report)
    print(json.dumps(accuracy_report, ensure_ascii=False, indent=2))
    display(FileLink(str(accuracy_path)))
